# AMI Subset ASR-Diarization Alignment (Whisper + pyannote)

This notebook aligns Whisper ASR segments with pyannote diarization turns and computes per-segment features for candidate selection.

In [1]:
from __future__ import annotations

import json
import math
import random
import re
from dataclasses import dataclass
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ASR_DIR = PROJECT_ROOT / "data" / "asr_outputs" / "segments"
DIAR_DIR = PROJECT_ROOT / "data" / "diarization_outputs" / "json"
ALIGNED_DIR = PROJECT_ROOT / "data" / "aligned_chunks"
METRICS_DIR = PROJECT_ROOT / "data" / "metrics"

ALIGNED_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

TOKEN_RE = re.compile(r"[a-z0-9]+")

random.seed(0)

In [2]:
@dataclass
class AsrSegment:
    start: float
    end: float
    text: str
    avg_logprob: float
    no_speech_prob: float


@dataclass
class DiarSegment:
    start: float
    end: float
    speaker: str
    confidence: float
    seg_consistency: float
    overlap: float
    flip_rate: float


def load_asr_segments(path: Path) -> list[AsrSegment]:
    with path.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)

    segments = []
    for row in payload.get("segments", []):
        segments.append(
            AsrSegment(
                start=float(row["start"]),
                end=float(row["end"]),
                text=str(row.get("text", "")),
                avg_logprob=float(row.get("avg_logprob", -10.0)),
                no_speech_prob=float(row.get("no_speech_prob", 0.0)),
            )
        )

    return segments


def load_diar_segments(path: Path) -> list[DiarSegment]:
    with path.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)

    segments = []
    for row in payload.get("segments", []):
        segments.append(
            DiarSegment(
                start=float(row["start"]),
                end=float(row["end"]),
                speaker=str(row.get("speaker", "UNKNOWN")),
                confidence=float(row.get("confidence", 0.0)),
                seg_consistency=float(row.get("seg_consistency", 0.0)),
                overlap=float(row.get("overlap", 0.0)),
                flip_rate=float(row.get("flip_rate", 0.0)),
            )
        )

    return segments


def compute_overlap(start_a: float, end_a: float, start_b: float, end_b: float) -> float:
    return max(0.0, min(end_a, end_b) - max(start_a, start_b))


def normalize_logprobs(segments: list[AsrSegment]) -> dict[int, float]:
    if not segments:
        return {}

    values = [seg.avg_logprob for seg in segments]
    min_val = min(values)
    max_val = max(values)
    span = max_val - min_val

    normalized = {}
    for idx, seg in enumerate(segments):
        if span == 0:
            normalized[idx] = 1.0
        else:
            normalized[idx] = (seg.avg_logprob - min_val) / span

    return normalized


def tokenize(text: str) -> set[str]:
    return set(TOKEN_RE.findall(text.lower()))


def jaccard(a: set[str], b: set[str]) -> float:
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


def compute_redundancy(asr_segments: list[AsrSegment], idx: int) -> float:
    current = tokenize(asr_segments[idx].text)
    neighbors = []
    if idx > 0:
        neighbors.append(tokenize(asr_segments[idx - 1].text))
    if idx + 1 < len(asr_segments):
        neighbors.append(tokenize(asr_segments[idx + 1].text))

    if not neighbors:
        return 0.0

    return max(jaccard(current, neighbor) for neighbor in neighbors)


def compute_turn_completeness(segment: AsrSegment) -> float:
    duration = max(0.0, segment.end - segment.start)
    duration_score = min(1.0, duration / 5.0)

    text = segment.text.strip()
    token_count = len(tokenize(text))
    if text.endswith((".", "?", "!")):
        punctuation_score = 1.0
    elif text.endswith(","):
        punctuation_score = 0.7
    else:
        punctuation_score = 0.85

    length_score = min(1.0, token_count / 6.0)
    speech_score = max(0.0, 1.0 - segment.no_speech_prob)

    return duration_score * punctuation_score * length_score * speech_score


def compute_diar_stability(overlaps: list[tuple[DiarSegment, float]]) -> float:
    if not overlaps:
        return 0.0

    total_overlap = sum(overlap for _, overlap in overlaps)
    if total_overlap <= 0.0:
        return 0.0

    speaker_overlap: dict[str, float] = {}
    for diar_seg, overlap in overlaps:
        speaker_overlap[diar_seg.speaker] = speaker_overlap.get(diar_seg.speaker, 0.0) + overlap

    dominant_overlap = max(speaker_overlap.values())
    speaker_consistency = dominant_overlap / total_overlap

    weighted_conf = 0.0
    weighted_consistency = 0.0
    weighted_non_overlap = 0.0
    weighted_non_flip = 0.0
    for diar_seg, overlap in overlaps:
        weight = overlap / total_overlap
        weighted_conf += diar_seg.confidence * weight
        weighted_consistency += diar_seg.seg_consistency * weight
        weighted_non_overlap += (1.0 - diar_seg.overlap) * weight
        weighted_non_flip += (1.0 - diar_seg.flip_rate) * weight

    stability = (
        speaker_consistency
        * weighted_conf
        * weighted_consistency
        * weighted_non_overlap
        * weighted_non_flip
    )
    return max(0.0, min(1.0, stability))

In [3]:
def align_asr_to_diar(asr_segments: list[AsrSegment], diar_segments: list[DiarSegment]) -> list[dict]:
    logprob_norm = normalize_logprobs(asr_segments)
    aligned_rows = []

    for idx, asr_seg in enumerate(asr_segments):
        overlaps = []
        for diar_seg in diar_segments:
            overlap = compute_overlap(
                asr_seg.start,
                asr_seg.end,
                diar_seg.start,
                diar_seg.end,
            )
            if overlap > 0.0:
                overlaps.append((diar_seg, overlap))

        if overlaps:
            overlaps.sort(
                key=lambda item: (
                    item[1],
                    item[0].confidence,
                    -item[0].start,
                ),
                reverse=True,
            )
            best_diar, best_overlap = overlaps[0]
            speaker = best_diar.speaker
            diar_turn_start = best_diar.start
            diar_turn_end = best_diar.end
        else:
            best_overlap = 0.0
            speaker = "UNKNOWN"
            diar_turn_start = math.nan
            diar_turn_end = math.nan

        asr_conf = logprob_norm.get(idx, 0.0)
        diar_stab = compute_diar_stability(overlaps)
        turn_comp = compute_turn_completeness(asr_seg)
        redund = compute_redundancy(asr_segments, idx)

        aligned_rows.append(
            {
                "start": asr_seg.start,
                "end": asr_seg.end,
                "text": asr_seg.text,
                "speaker": speaker,
                "ASRConf": asr_conf,
                "DiarStab": diar_stab,
                "TurnComp": turn_comp,
                "Redund": redund,
                "diar_turn_start": diar_turn_start,
                "diar_turn_end": diar_turn_end,
                "diar_overlap_sec": best_overlap,
            }
        )

    return aligned_rows


def process_file(
    file_id: str,
    asr_dir: Path,
    diar_dir: Path,
    aligned_dir: Path,
    metrics_dir: Path,
) -> dict[str, float]:
    asr_path = asr_dir / f"{file_id}.json"
    diar_path = diar_dir / f"{file_id}.json"
    if not asr_path.exists() or not diar_path.exists():
        return {"processed": 0.0}

    asr_segments = load_asr_segments(asr_path)
    diar_segments = load_diar_segments(diar_path)
    aligned_rows = align_asr_to_diar(asr_segments, diar_segments)

    aligned_df = pd.DataFrame(aligned_rows)
    aligned_df.insert(0, "file_id", file_id)

    aligned_out = aligned_dir / f"{file_id}_aligned.csv"
    aligned_df.to_csv(aligned_out, index=False)

    metrics_df = aligned_df[
        [
            "file_id",
            "start",
            "end",
            "speaker",
            "ASRConf",
            "DiarStab",
            "TurnComp",
            "Redund",
        ]
    ]
    metrics_out = metrics_dir / f"{file_id}_features.csv"
    metrics_df.to_csv(metrics_out, index=False)

    return {"processed": float(len(aligned_df))}


def collect_file_ids(asr_dir: Path, diar_dir: Path) -> list[str]:
    asr_ids = {path.stem for path in asr_dir.glob("*.json")}
    diar_ids = {path.stem for path in diar_dir.glob("*.json")}
    return sorted(asr_ids & diar_ids)

In [4]:
file_ids = collect_file_ids(ASR_DIR, DIAR_DIR)

processed = 0
for file_id in file_ids:
    result = process_file(file_id, ASR_DIR, DIAR_DIR, ALIGNED_DIR, METRICS_DIR)
    processed += int(result.get("processed", 0.0))

summary_df = pd.DataFrame({
    "metric": ["files", "segments"],
    "value": [len(file_ids), processed],
})
summary_df

,metric,value
0,files,12
1,segments,6405
